# 🃏 Universal Poker YOLOv8 — работает на ЛЮБОМ столе

Стратегия обобщения:
- Скачиваем **публичные датасеты** покера с Roboflow Universe (тысячи размеченных скриншотов из разных румов)
- Добавляем свои кадры с GoP3
- Обучаем с **агрессивной аугментацией по цвету** — модель не запоминает зелёный стол, а учится находить структуру

**Порядок:** `Runtime → T4 GPU → Run All`

**Ожидаемое время:** ~40–60 минут

## 1. Установка

In [ ]:
!pip install ultralytics roboflow onnx onnxruntime -q
import torch
print(f'PyTorch: {torch.__version__}')
print(f'GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else "❌ НЕТ — смени Runtime на T4!"}')

## 2. Скачать публичные покерные датасеты с Roboflow Universe

Roboflow хранит тысячи датасетов от разных пользователей. Бесплатно.
Здесь берём самые полезные для карт + покерного стола.

In [ ]:
# Получи бесплатный API-ключ на roboflow.com (регистрация 30 секунд)
RF_API_KEY = ""  # <-- вставь сюда свой ключ

from roboflow import Roboflow
from pathlib import Path
import shutil

RF_DIR = Path('/content/rf_datasets')
RF_DIR.mkdir(exist_ok=True)

rf = Roboflow(api_key=RF_API_KEY)

# Список публичных датасетов покерных карт/столов
# Каждый — тысячи скриншотов из разных источников
PUBLIC_DATASETS = [
    # (workspace, project, version)
    ("roboflow-100",  "playing-cards-ow3dr",         4),   # 20 000+ карт, разные фоны
    ("poker-xkf8m",   "poker-cards-detection",        2),   # реальные столы
    ("datasetsroboflow", "playing-cards-detection",   1),   # смешанные стили
]

downloaded = []
for ws, proj, ver in PUBLIC_DATASETS:
    try:
        print(f'⬇️  {proj}...')
        project = rf.workspace(ws).project(proj)
        dataset = project.version(ver).download("yolov8", location=str(RF_DIR / proj))
        downloaded.append(RF_DIR / proj)
        print(f'   ✅ Скачан')
    except Exception as e:
        print(f'   ⚠️  Пропускаю {proj}: {e}')

print(f'\nСкачано датасетов: {len(downloaded)}')

## 3. Загрузить свои GoP3 кадры (опционально)

In [ ]:
from google.colab import files
import zipfile
from pathlib import Path

print('Загрузи zip с GoP3 кадрами (или пропусти — Ctrl+C)')
print('Структура: images/ + labels/ + data.yaml')

try:
    uploaded = files.upload()
    if uploaded:
        zip_name = list(uploaded.keys())[0]
        custom_dir = Path('/content/rf_datasets/gop3_custom')
        custom_dir.mkdir(exist_ok=True)
        with zipfile.ZipFile(zip_name, 'r') as z:
            z.extractall(custom_dir)
        imgs = list(custom_dir.rglob('*.jpg')) + list(custom_dir.rglob('*.png'))
        print(f'✅ GoP3: {len(imgs)} кадров добавлено')
except KeyboardInterrupt:
    print('Пропускаю свой датасет — используем только публичные.')

## 4. Слияние всех датасетов в один

In [ ]:
import random, yaml, shutil
from pathlib import Path

# Классы — стандартные 52 карты (совместимо с playing_cards.onnx)
# + новые классы покерного стола
CARD_CLASSES = [
    '10C','10D','10H','10S','2C','2D','2H','2S','3C','3D','3H','3S',
    '4C','4D','4H','4S','5C','5D','5H','5S','6C','6D','6H','6S',
    '7C','7D','7H','7S','8C','8D','8H','8S','9C','9D','9H','9S',
    'AC','AD','AH','AS','JC','JD','JH','JS','KC','KD','KH','KS',
    'QC','QD','QH','QS'
]
TABLE_CLASSES = ['dealer_button', 'player_folded', 'player_away', 'pot_chips', 'player_bet', 'stack_label']
ALL_CLASSES   = CARD_CLASSES + TABLE_CLASSES
CLASS_MAP     = {c: i for i, c in enumerate(ALL_CLASSES)}

MERGED = Path('/content/merged')
for split in ['train', 'val']:
    (MERGED / 'images' / split).mkdir(parents=True, exist_ok=True)
    (MERGED / 'labels' / split).mkdir(parents=True, exist_ok=True)

def remap_label_file(src: Path, dst: Path, src_names: list[str]):
    """Переписывает индексы классов под общий словарь."""
    lines_out = []
    with open(src) as f:
        for line in f:
            parts = line.strip().split()
            if not parts: continue
            old_idx = int(parts[0])
            if old_idx >= len(src_names): continue
            cls_name = src_names[old_idx]
            # Нормализуем имя карты к нашему формату
            cls_name = cls_name.upper().replace('_OF_', '').replace(' ', '')
            # Примеры нормализации из разных датасетов:
            # 'ace_of_hearts' -> 'AH', 'king-clubs' -> 'KC'
            for rank in ['10','A','K','Q','J','2','3','4','5','6','7','8','9']:
                for suit, letter in [('HEARTS','H'),('DIAMONDS','D'),('CLUBS','C'),('SPADES','S')]:
                    if rank in cls_name and suit in cls_name:
                        cls_name = rank + letter
                        break
            new_idx = CLASS_MAP.get(cls_name)
            if new_idx is None: continue
            lines_out.append(f"{new_idx} {' '.join(parts[1:])}")
    if lines_out:
        dst.write_text('\n'.join(lines_out))

total = 0
for ds_dir in Path('/content/rf_datasets').iterdir():
    # Найдём data.yaml в датасете
    yaml_files = list(ds_dir.rglob('data.yaml'))
    if not yaml_files: continue
    with open(yaml_files[0]) as f:
        ds_yaml = yaml.safe_load(f)
    src_names = ds_yaml.get('names', [])

    for split in ['train', 'valid', 'val', 'test']:
        out_split = 'val' if split in ('val','valid','test') else 'train'
        img_dir = ds_dir / split / 'images'
        lbl_dir = ds_dir / split / 'labels'
        if not img_dir.exists():
            img_dir = ds_dir / 'images' / split
            lbl_dir = ds_dir / 'labels' / split
        if not img_dir.exists(): continue

        for img_path in img_dir.glob('*'):
            if img_path.suffix.lower() not in ('.jpg','.jpeg','.png'): continue
            lbl_path = (lbl_dir / img_path.stem).with_suffix('.txt')
            if not lbl_path.exists(): continue
            stem = f"{ds_dir.name}_{img_path.stem}"
            dst_img = MERGED / 'images' / out_split / (stem + img_path.suffix)
            dst_lbl = MERGED / 'labels' / out_split / (stem + '.txt')
            shutil.copy(img_path, dst_img)
            remap_label_file(lbl_path, dst_lbl, src_names)
            total += 1

train_imgs = list((MERGED / 'images' / 'train').glob('*'))
val_imgs   = list((MERGED / 'images' / 'val').glob('*'))

# Если val пустой — откусываем 10% от train
if len(val_imgs) < 10:
    random.shuffle(train_imgs)
    for p in train_imgs[:max(10, len(train_imgs)//10)]:
        lbl = (MERGED / 'labels' / 'train' / p.stem).with_suffix('.txt')
        shutil.move(str(p),   MERGED / 'images' / 'val' / p.name)
        if lbl.exists(): shutil.move(str(lbl), MERGED / 'labels' / 'val' / lbl.name)

# data.yaml
with open(MERGED / 'data.yaml', 'w') as f:
    yaml.dump({'path': str(MERGED), 'train': 'images/train', 'val': 'images/val',
               'nc': len(ALL_CLASSES), 'names': ALL_CLASSES}, f, allow_unicode=True, sort_keys=False)

train_count = len(list((MERGED / 'images' / 'train').glob('*')))
val_count   = len(list((MERGED / 'images' / 'val').glob('*')))
print(f'\n✅ Датасет собран: {train_count} train / {val_count} val ({len(ALL_CLASSES)} классов)')
print(f'   Карты: 0–51  |  Стол: 52–{len(ALL_CLASSES)-1}')

## 5. Тренировка с агрессивной аугментацией

Ключ к универсальности — **сильные цветовые аугментации**.
Модель видит одни и те же карты на зелёном, синем, тёмном, светлом столах.
После этого цвет стола перестаёт быть признаком.

In [ ]:
from ultralytics import YOLO

model = YOLO('yolov8n.pt')   # быстрый; замени на yolov8s.pt для +качество

results = model.train(
    data    = '/content/merged/data.yaml',
    epochs  = 80,
    imgsz   = 640,
    batch   = 16,
    device  = 0,
    workers = 2,
    patience= 20,
    name    = 'universal_poker',
    project = '/content/runs',

    # ── Цветовая инвариантность ────────────────────────────────────────────
    # Случайный сдвиг оттенка на ±50% — модель видит все цвета стола
    hsv_h   = 0.5,
    # Случайное насыщение ×(0.3..1.7) — и яркий и тусклый стол
    hsv_s   = 0.7,
    # Яркость ×(0.6..1.4) — ночной/дневной экран
    hsv_v   = 0.4,

    # ── Геометрическая инвариантность ─────────────────────────────────────
    # Разные зумы — мобильный/планшет/монитор
    scale   = 0.5,
    # Лёгкая перспектива — наклон экрана
    perspective = 0.001,
    # Мозаика 4-в-1 — больше контекстов за эпоху
    mosaic  = 1.0,
    # Кроп и паддинг
    translate = 0.2,

    # ── Карты не переворачиваем ────────────────────────────────────────────
    flipud  = 0.0,   # вертикальный флип — нет
    fliplr  = 0.5,   # горизонтальный — ok, стол симметричен

    # ── Дополнительно ─────────────────────────────────────────────────────
    # Случайно стираем части изображения — учим не зависеть от одной детали
    erasing = 0.3,
    # Размытие — разное качество экрана/камеры
    blur    = 0.1,
)

print(f'\n✅ Готово! mAP50: {results.results_dict.get("metrics/mAP50(B)", "?")}')

## 6. Экспорт в ONNX

In [ ]:
from ultralytics import YOLO
import glob, shutil
from pathlib import Path

candidates = glob.glob('/content/runs/**/best.pt', recursive=True)
best_pt    = Path(sorted(candidates)[-1]) if candidates else None

if best_pt:
    print(f'Экспортирую: {best_pt}')
    YOLO(str(best_pt)).export(format='onnx', imgsz=640, opset=12, simplify=True, dynamic=False)
    shutil.copy(best_pt.with_suffix('.onnx'), '/content/universal_poker.onnx')
    import os
    mb = os.path.getsize('/content/universal_poker.onnx') / 1024 / 1024
    print(f'✅ /content/universal_poker.onnx  ({mb:.1f} MB)')
else:
    print('❌ best.pt не найден')

## 7. Тест на произвольном скриншоте

In [ ]:
from ultralytics import YOLO
from PIL import Image
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from google.colab import files

CARD_CLASSES  = ['10C','10D','10H','10S','2C','2D','2H','2S','3C','3D','3H','3S','4C','4D','4H','4S','5C','5D','5H','5S','6C','6D','6H','6S','7C','7D','7H','7S','8C','8D','8H','8S','9C','9D','9H','9S','AC','AD','AH','AS','JC','JD','JH','JS','KC','KD','KH','KS','QC','QD','QH','QS']
TABLE_CLASSES = ['dealer_button','player_folded','player_away','pot_chips','player_bet','stack_label']
ALL_CLASSES   = CARD_CLASSES + TABLE_CLASSES

print('Загрузи любой скриншот покера для проверки...')
uploaded = files.upload()
test_img = list(uploaded.keys())[0]

model = YOLO('/content/universal_poker.onnx', task='detect')
res   = model(test_img, conf=0.35)[0]

img = Image.open(test_img)
fig, ax = plt.subplots(figsize=(14, 8))
ax.imshow(img)
for box in res.boxes:
    x1,y1,x2,y2 = box.xyxy[0].tolist()
    cls   = ALL_CLASSES[int(box.cls)]
    conf  = float(box.conf)
    color = '#00ff88' if int(box.cls) < 52 else '#ff4444'
    ax.add_patch(patches.Rectangle((x1,y1),x2-x1,y2-y1,linewidth=2,edgecolor=color,facecolor='none'))
    ax.text(x1,y1-4,f'{cls} {conf:.2f}',color=color,fontsize=8,
            bbox=dict(facecolor='black',alpha=0.5,pad=1))
ax.axis('off')
plt.title(f'Детекций: {len(res.boxes)}')
plt.savefig('/content/test_result.jpg', dpi=150, bbox_inches='tight')
plt.show()

## 8. Скачать модель

In [ ]:
from google.colab import files
files.download('/content/universal_poker.onnx')
print('Положи файл в: artifacts/poker-advisor/public/models/universal_poker.onnx')